In [1]:
import tensorflow as tf
import TensorSlider as ts
import keras
import numpy as np
import DataPrep


2025-02-09 17:06:27.888973: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739117187.911287  190540 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739117187.916992  190540 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-09 17:06:27.942921: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Get Datasets

In [3]:
tfrecordpath = "../Data/tfrecords/"

windowsize = 150
lookahead = 5
batch_size = 20

coins = ["BTCUSD_PERP", "ETHUSD_PERP", "ADAUSD_PERP", "SOLUSD_PERP", "BNBUSDT_PERP", "LINKUSD_PERP", "TRXUSD_PERP", "XLMUSD_PERP","DOTUSD_PERP"]
datasets = DataPrep.getAllSliders(coins, windowsize, lookahead, batch_size, tfrecordpath)
valdataset = datasets.pop(-1)

I0000 00:00:1739117191.020318  190540 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1817 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:05:00.0, compute capability: 8.6


## Combine Datasets

In [4]:
import keras
import os

def load_model(name, optimizer):
    """
    Load model and latest checkpoint (if applicable). Returns model, and last epoch that was trained.
    If no checkpoints present, either creates the checkpoint folder or trains directly on the saved model.
    """
    folder = "models/" + name + "/"
    # Load model
    model = keras.models.load_model(folder + "model.keras")

    checkpoint = tf.train.Checkpoint(optimizer=optimizer, model=model)
    manager = tf.train.CheckpointManager(checkpoint=checkpoint, directory=folder, max_to_keep=3)
    manager.restore_or_initialize()

    return model, manager

class saveEachEpoch(tf.keras.callbacks.Callback):
    """
    custom callback for saving checkpoints because of course we need to do this on our own
    """

    def __init__(self, checkpointmanager):
        super().__init__()
        # no idea if we want to or need to super this
        try:
            self.lastEpoch = int(checkpointmanager.latest_checkpoint.split("-")[-1])
        except Exception as e:
            print(e)
            self.lastEpoch = 0

        self.checkpointManager = checkpointmanager
        print("Model was trained for " + str(self.lastEpoch) + " epochs before.")

    def on_epoch_end(self, epoch, logs):
        """
        Create a checkpoint and save.
        """
        print(f"Epoch {self.lastEpoch} ended")
        # Increment which epoch this is
        self.lastEpoch += 1
        # Save weights
        self.checkpointManager.save(checkpoint_number=epoch)


In [5]:
#zip the different dataset sources
zipped = tf.data.Dataset.zip(datasets=tuple(datasets))
def combineZippedBatches(*zipped):
    batchshape = zipped[0][0]
    # Create first tensors to concat the rest
    data = zipped[0][0]
    label = zipped[0][1]
    for i in range(1, len(zipped)): # iterate over each remaining pair
        data = tf.concat([data, zipped[i][0]], axis=0)
        label = tf.concat([label, zipped[i][1]], axis=0)

    return data, label

# combine batches into one megabatch
batchTogether = zipped.map(combineZippedBatches)

# Prefetch and create labels
training = batchTogether.map(DataPrep.createLabelsBatch, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
validation = valdataset.map(DataPrep.createLabelsBatch, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

In [ ]:
modelName = "dingus2"

tensorboard = keras.callbacks.TensorBoard(
                                                log_dir=f"models/{modelName}/logs",
                                                histogram_freq=100,
                                                write_graph=True,
                                                write_images=False,
                                                write_steps_per_second=True,
                                                update_freq="batch",
                                                #profile_batch = '70,100',
                                                embeddings_freq=0,
                                                embeddings_metadata=None,
                                            )
optimizer = keras.optimizers.Adam(amsgrad=True, clipvalue=0.2)

model, manager = load_model(modelName, optimizer)

model.compile(loss=keras.losses.mean_squared_error,
                  optimizer=optimizer,
                  # For the metrics, always have AT LEAST these two
                  metrics=["MeanAbsolutePercentageError", "MeanSquaredError"])

model.summary()

history = model.fit(training, epochs=5, verbose=0, validation_data=validation, callbacks=[saveEachEpoch(manager), tensorboard])

/root/miniconda3/envs/levbot/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 50 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_3 (Flatten)             │ (None, 4500)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 6000)           │    27,006,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 6000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ (None, 4000)           │    24,004,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 4000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_36 (Dense)                │ (None, 3000)           │    12,003,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 3000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ (None, 1000)           │     3,001,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_38 (Dense)                │ (None, 500)            │       500,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_22 (Dropout)            │ (None, 500)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_39 (Dense)                │ (None, 50)             │        25,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_40 (Dense)                │ (None, 20)             │         1,020 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ (None, 2)              │            42 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 66,540,612 (253.83 MB)

 Trainable params: 66,540,612 (253.83 MB)

 Non-trainable params: 0 (0.00 B)

Model was trained for 4 epochs before.


2025-02-09 17:06:38.354689: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:370] TFRecordDataset `buffer_size` is unspecified, default to 262144
I0000 00:00:1739117216.235118  190601 service.cc:148] XLA service 0x7f2ffc0145a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1739117216.235200  190601 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 3060 Ti, Compute Capability 8.6
2025-02-09 17:06:56.342521: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1739117216.525251  190601 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1739117218.501889  190601 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2025-02-09 17:10:20.127050: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status:

Epoch 4 ended


2025-02-09 17:10:53.788530: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 6480000000 exceeds 10% of free system memory.
2025-02-09 17:10:55.084032: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 5760000000 exceeds 10% of free system memory.
2025-02-09 17:10:56.102017: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 2880000000 exceeds 10% of free system memory.
